## Parser

In [3]:
import os
import time
import requests
from urllib.parse import quote
from typing import Any

GITHUB_TOKEN = 'SOME TOKEN'
if not GITHUB_TOKEN:
    raise ValueError("Необходимо задать GITHUB_TOKEN в переменных окружения")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
TARGET_LICENSES = {"mit", "cc0-1.0", None}

DATASET_DIR = "Dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

SEARCH_QUERY = "topic:python-projects language:python"
MAX_PER_PAGE = 1000
MAX_PAGES = 2
REQUEST_DELAY = 0.5

def search_repositories(query, page=1) -> tuple[Any, Any]:
    url = "https://api.github.com/search/repositories"
    params = {
        "q": query,
        "per_page": MAX_PER_PAGE,
        "page": page
    }
    resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["items"], data["total_count"]

def get_default_branch(owner, repo) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["default_branch"]

def get_repo_tree(owner, repo, branch) -> Any:
    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()["tree"]

def get_file_content(owner, repo, file_path, branch) -> Any | str | None:
    path_encoded = quote(file_path)
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path_encoded}?ref={branch}"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code == 403 and "rate limit" in resp.text.lower():
        print("Request limit exceeded! Waiting")
        time.sleep(60)
        return get_file_content(owner, repo, file_path, branch)
    if resp.status_code != 200:
        print(f"Unable to get file {file_path} (status {resp.status_code})")
        return None
    data = resp.json()
    if data["type"] != "file":
        return None
    download_url = data["download_url"]
    file_resp = requests.get(download_url, headers=HEADERS)
    if file_resp.status_code != 200:
        print(f"Unable to load file: {file_path}")
        return None
    return file_resp.text

def save_file(content, repo_full_name, file_path) -> None:
    safe_name = f"{repo_full_name.replace('/', '__')}__{file_path.replace('/', '_')}"
    if len(safe_name) > 250:
        safe_name = safe_name[:250] + ".py"
    filepath = os.path.join(DATASET_DIR, safe_name)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Saved: {safe_name}")


def main() -> None:
    page = 1
    total_downloaded = 0
    processed_repos = 0

    while page <= MAX_PAGES:
        print(f"Loading {page}...")
        try:
            items, total_count = search_repositories(SEARCH_QUERY, page)
        except Exception as e:
            print(f"Error while searching: {e}")
            break

        if not items:
            print("No repos on the page")
            break

        print(f"Found repos on the page: {len(items)} (sum: {total_count})")

        for repo in items:
            repo_full_name = repo["full_name"]
            license_info = repo.get("license")
            license_key = license_info["key"] if license_info else None

            if license_key not in TARGET_LICENSES:
                continue
            try:
                branch = get_default_branch(*repo_full_name.split('/'))
                time.sleep(REQUEST_DELAY)

                tree = get_repo_tree(*repo_full_name.split('/'), branch)
                time.sleep(REQUEST_DELAY)

                py_files = [item for item in tree if item["type"] == "blob" and item["path"].endswith(".py")]
                if not py_files:
                    print(" No .py files")
                    continue
                print(f"Found .py files: {len(py_files)}")

                for file_item in py_files:
                    file_path = file_item["path"]
                    content = get_file_content(*repo_full_name.split('/'), file_path, branch)
                    if content is not None:
                        save_file(content, repo_full_name, file_path)
                        total_downloaded += 1
                    time.sleep(REQUEST_DELAY)

            except Exception as e:
                print(f"Error while parsing repo: {e}")

            processed_repos += 1

        if len(items) < MAX_PER_PAGE:
            break 
        page += 1
        time.sleep(REQUEST_DELAY * 2) 

    print()
    print(f"Done! Parsed repos: {processed_repos}, files downloaded: {total_downloaded}")

main()

Loading 1...
Found repos on the page: 100 (sum: 310)
Found .py files: 33
Saved: qxresearch__qxresearch-event-1__Applications_Alarm_alarmtiming.py
Saved: qxresearch__qxresearch-event-1__Applications_Audio Visualization Tool_source-code.py
Saved: qxresearch__qxresearch-event-1__Applications_Birthday Reminder_source-code.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_1_FreshProject.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_1_GETdata.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_2_FreshProject1.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_2_recordAudio.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_3_FreshProject2.py
Saved: qxresearch__qxresearch-event-1__Applications_CSPRNG_CSPRNG_4_FreshProject3.py
Saved: qxresearch__qxresearch-event-1__Applications_Calendar_calendar.py
Saved: qxresearch__qxresearch-event-1__Applications_Extract mp3 from mp4_source-code.py
Saved: qxresear

In [4]:
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [11]:
FILE_DOWNLOAD_PATH = r'C:\Users\Roman\Documents\Labs\bfu_nlp_labs\Final_Work\Dataset'
# FILE_PATH = 'C:\Users\Roman\Documents\Labs\bfu_nlp_labs\Final_Work\Dataset'
# for filename in os.listdir(FILE_PATH):
#     (filename)

In [29]:
text_sample = []
for filename in os.listdir(FILE_DOWNLOAD_PATH):
    with open(FILE_DOWNLOAD_PATH + '/' + filename, 'r', encoding='utf-8') as file:
        text_sample.extend(file.readlines())

print('Length of text sample before redaction: ', len(text_sample))

i = 0
while i < len(text_sample):
    if text_sample[i] != '\n':
        i += 1
        continue
    text_sample.pop(i)
print('Length of text sample after redaction: ', len(text_sample))
text_sample = ' '.join(text_sample)

def text_to_seq(text_sample):
    char_counts = Counter(text_sample)
    char_counts = sorted(char_counts.items(), key = lambda x: x[1], reverse=True)

    sorted_chars = [char for char, _ in char_counts]
    print(sorted_chars)
    char_to_idx = {char: index for index, char in enumerate(sorted_chars)}
    idx_to_char = {v: k for k, v in char_to_idx.items()}
    sequence = np.array([char_to_idx[char] for char in text_sample])
    
    return sequence, char_to_idx, idx_to_char

sequence, char_to_idx, idx_to_char = text_to_seq(text_sample)

Length of text sample before redaction:  808952
Length of text sample after redaction:  664577
[' ', 'e', 't', 'r', 's', 'i', 'a', 'n', 'o', '\n', 'l', ',', 'd', 'c', 'p', 'f', 'u', "'", 'm', '_', '"', '.', ')', '(', 'h', 'x', '0', ':', 'g', '=', 'b', '1', 'y', '#', '2', '\\', '3', 'w', 'v', '5', 'k', '4', '-', 'E', '6', '8', 'T', '7', '9', 'S', 'A', 'C', 'N', 'R', 'I', 'M', 'F', 'P', 'D', 'O', 'L', 'q', '[', ']', 'j', 'B', '+', '*', 'U', 'H', 'z', '`', 'W', 'V', '{', 'G', '>', '%', '}', 'X', 'Y', 'K', ';', 'Q', 'Z', '/', '<', 'J', '!', '?', '@', '|', '^', '$', '&', '~', '\t', '█', 'ı', 'م', 'а', 'т', 'о', 'и', 'е', 'ι', 'м', 'н', 'с', 'к', 'л', 'д', 'ü', 'б', 'в', 'ь', 'з', 'у', 'ي', 'р', 'ö', 'я', 'п', 'ю', 'ц', 'ч', 'В', 'х', 'й', 'г', 'Г', 'И', 'У', 'ж', 'ф', 'Д', 'Ч', 'ш', 'ъ', 'С', 'З', 'Н', 'О', 'П', 'щ', 'А', 'Б', 'Е', 'Ж', 'К', 'Л', 'М', 'Р', 'Т', 'Ф', 'Х', 'Ц', 'Ш', 'Й', 'Ю', 'Я', 'Щ', 'Ö', 'Ü', '·', 'ح', 'ج', 'é', 'ş', '─', 'ğ', '\u2001', 'θ', 'ç', 'π', 'ּ', 'ы', 'μ', 'ى', '

In [30]:
print(char_to_idx)

{' ': 0, 'e': 1, 't': 2, 'r': 3, 's': 4, 'i': 5, 'a': 6, 'n': 7, 'o': 8, '\n': 9, 'l': 10, ',': 11, 'd': 12, 'c': 13, 'p': 14, 'f': 15, 'u': 16, "'": 17, 'm': 18, '_': 19, '"': 20, '.': 21, ')': 22, '(': 23, 'h': 24, 'x': 25, '0': 26, ':': 27, 'g': 28, '=': 29, 'b': 30, '1': 31, 'y': 32, '#': 33, '2': 34, '\\': 35, '3': 36, 'w': 37, 'v': 38, '5': 39, 'k': 40, '4': 41, '-': 42, 'E': 43, '6': 44, '8': 45, 'T': 46, '7': 47, '9': 48, 'S': 49, 'A': 50, 'C': 51, 'N': 52, 'R': 53, 'I': 54, 'M': 55, 'F': 56, 'P': 57, 'D': 58, 'O': 59, 'L': 60, 'q': 61, '[': 62, ']': 63, 'j': 64, 'B': 65, '+': 66, '*': 67, 'U': 68, 'H': 69, 'z': 70, '`': 71, 'W': 72, 'V': 73, '{': 74, 'G': 75, '>': 76, '%': 77, '}': 78, 'X': 79, 'Y': 80, 'K': 81, ';': 82, 'Q': 83, 'Z': 84, '/': 85, '<': 86, 'J': 87, '!': 88, '?': 89, '@': 90, '|': 91, '^': 92, '$': 93, '&': 94, '~': 95, '\t': 96, '█': 97, 'ı': 98, 'م': 99, 'а': 100, 'т': 101, 'о': 102, 'и': 103, 'е': 104, 'ι': 105, 'м': 106, 'н': 107, 'с': 108, 'к': 109, 'л': 1

In [31]:
print(idx_to_char)

{0: ' ', 1: 'e', 2: 't', 3: 'r', 4: 's', 5: 'i', 6: 'a', 7: 'n', 8: 'o', 9: '\n', 10: 'l', 11: ',', 12: 'd', 13: 'c', 14: 'p', 15: 'f', 16: 'u', 17: "'", 18: 'm', 19: '_', 20: '"', 21: '.', 22: ')', 23: '(', 24: 'h', 25: 'x', 26: '0', 27: ':', 28: 'g', 29: '=', 30: 'b', 31: '1', 32: 'y', 33: '#', 34: '2', 35: '\\', 36: '3', 37: 'w', 38: 'v', 39: '5', 40: 'k', 41: '4', 42: '-', 43: 'E', 44: '6', 45: '8', 46: 'T', 47: '7', 48: '9', 49: 'S', 50: 'A', 51: 'C', 52: 'N', 53: 'R', 54: 'I', 55: 'M', 56: 'F', 57: 'P', 58: 'D', 59: 'O', 60: 'L', 61: 'q', 62: '[', 63: ']', 64: 'j', 65: 'B', 66: '+', 67: '*', 68: 'U', 69: 'H', 70: 'z', 71: '`', 72: 'W', 73: 'V', 74: '{', 75: 'G', 76: '>', 77: '%', 78: '}', 79: 'X', 80: 'Y', 81: 'K', 82: ';', 83: 'Q', 84: 'Z', 85: '/', 86: '<', 87: 'J', 88: '!', 89: '?', 90: '@', 91: '|', 92: '^', 93: '$', 94: '&', 95: '~', 96: '\t', 97: '█', 98: 'ı', 99: 'م', 100: 'а', 101: 'т', 102: 'о', 103: 'и', 104: 'е', 105: 'ι', 106: 'м', 107: 'н', 108: 'с', 109: 'к', 110: '

In [32]:
SEQ_LEN = 16
BATCH_SIZE = 16

def get_batch(sequence):
    trains = []
    targets = []
    for _ in range(BATCH_SIZE):
        batch_start = np.random.randint(0, len(sequence) - SEQ_LEN)
        chunk = sequence[batch_start: batch_start + SEQ_LEN]
        train = torch.LongTensor(chunk[:-1]).view(-1, 1)
        target = torch.LongTensor(chunk[1:]).view(-1, 1)
        trains.append(train)
        targets.append(target)
    return torch.stack(trains, dim=0), torch.stack(targets, dim=0)

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

def evaluate(model, char_to_idx, idx_to_char, start_text=' ', prediction_len=200, temp=0.3):
    hidden = model.init_hidden()
    idx_input = [char_to_idx[char] for char in start_text]
    train = torch.LongTensor(idx_input).view(-1, 1, 1).to(device)
    predicted_text = start_text
    
    _, hidden = model(train, hidden)
        
    inp = train[-1].view(-1, 1, 1)
    
    for i in range(prediction_len):
        output, hidden = model(inp.to(device), hidden)
        output_logits = output.cpu().data.view(-1)
        p_next = F.softmax(output_logits / temp, dim=-1).detach().cpu().data.numpy()        
        top_index = np.random.choice(len(char_to_idx), p=p_next)
        inp = torch.LongTensor([top_index]).view(-1, 1, 1).to(device)
        predicted_char = idx_to_char[top_index]
        predicted_text += predicted_char
    
    return predicted_text

cpu


In [34]:
class TextRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, embedding_size, n_layers=1):
        super(TextRNN, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size
        self.n_layers = n_layers

        self.encoder = nn.Embedding(self.input_size, self.embedding_size)
        self.lstm = nn.LSTM(self.embedding_size, self.hidden_size, self.n_layers)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.hidden_size, self.input_size)
        
    def forward(self, x, hidden):
        x = self.encoder(x).squeeze(2)
        out, (ht1, ct1) = self.lstm(x, hidden)
        out = self.dropout(out)
        x = self.fc(out)
        return x, (ht1, ct1)
    
    def init_hidden(self, batch_size=1):
        return (torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device),
               torch.zeros(self.n_layers, batch_size, self.hidden_size, requires_grad=True).to(device))

In [35]:
%%time
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#model = TextRNN(input_size=len(idx_to_char), hidden_size=128, embedding_size=128, n_layers=2)
model = TextRNN(input_size=len(idx_to_char), hidden_size=128, embedding_size=64, n_layers=4)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    patience=5, 
    factor=0.5
)

n_epochs = 5000 #50000
loss_avg = []

for epoch in range(n_epochs):
    model.train()
    train, target = get_batch(sequence)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(BATCH_SIZE)

    output, hidden = model(train, hidden)
    loss = criterion(output.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    loss_avg.append(loss.item())
    if len(loss_avg) >= 50:
        mean_loss = np.mean(loss_avg)
        print(f'Loss: {mean_loss}')
        scheduler.step(mean_loss)
        loss_avg = []
        model.eval()
        predicted_text = evaluate(model, char_to_idx, idx_to_char)
        print(predicted_text)

Loss: 3.994807662963867
              a   e                                         e e                         x                                                                    0                                           
Loss: 3.6322513246536254
                                                                                                                                                              e             
                            
Loss: 3.6207776355743406
            e        i    tt     s i           
                          i i     e   s  i t       si t e tt  n    r  t t r  l       r  t          t       t t at          s  s                        n 
Loss: 3.5548006105422973
                                                    r           t                                                                                                                                        
Loss: 3.6244125318527223
    t   n  p                                                ,     r 